# 03 — Generative UI

**Agents that design their own surface.**

Instead of returning a blob of text, an agent can emit a *typed UI component* — a chart, a table, a plan, a form. The component is a Pydantic spec; it travels over the `EventBus` as a `ui.<type>.init` event; the frontend resolves a renderer by `component_type` and draws it. Subsequent state changes flow as `ui.<type>.delta` events.

This notebook composes: `Workbench` (carrying the `ComponentRegistry` + `EventBus`) + a real agent that produces data + `UIEmitter` + `ExecutionPlan` + `sse_sidecar`.

**Prerequisites:** a `.env` with `LLM_API_KEY` and `LLM_MODEL`.

## 1. Load config

In [1]:
from orqest import load_config

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. The Workbench carries the component registry

A `Workbench` bundles the five runtime pieces — memory, tracer, event bus, a recent-events buffer, and a `ComponentRegistry`. By default the registry is pre-loaded with the first-party components: 17 of them, across three layers (compositional primitives, declarative grammars, and a sandboxed HTML escape hatch).

In [2]:
import tempfile, os
from orqest.workbench import Workbench
from orqest.memory import LocalMemoryStore

wb = Workbench(memory=LocalMemoryStore(os.path.join(tempfile.mkdtemp(), "ui.db")))

types = wb.ui_registry.list_types()
print(f"{len(types)} first-party component types:")
print("  " + ", ".join(types))

17 first-party component types:
  badge, button, chart, form, image, input, json_viewer, latex, layout, markdown, mermaid, plan, sandboxed_html, table, takeover_dialog, text, vega_chart


## 3. An agent produces structured data

The agent's job is the *content*; the UI layer is the *surface*. Here a plain `BaseAgent` turns a topic into a small labelled dataset — exactly the shape a chart wants.

In [3]:
from pydantic import BaseModel, Field
from orqest.agents import BaseAgent, GlobalState


class DataPoint(BaseModel):
    label: str
    value: float


class Dataset(BaseModel):
    title: str = Field(description="A short chart title.")
    series_name: str = Field(description="What the values measure.")
    points: list[DataPoint] = Field(description="4-6 labelled data points.")


class DataAgent(BaseAgent[GlobalState, Dataset]):
    def __init__(self, model, api_key=None):
        super().__init__(
            agent_name="data_agent",
            system_prompt=(
                "You turn a topic into a small, plausible labelled dataset "
                "suitable for a bar chart. 4-6 points. Values are illustrative."
            ),
            output_type=Dataset,
            model=model,
            api_key=api_key,
        )

    async def _run_implementation(self, state: GlobalState, **kwargs) -> Dataset:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


data_agent = DataAgent(model=config.llm_model, api_key=config.llm_api_key)

state = GlobalState()
state.add_message("user", "Approximate share of developers using each major code editor.")
dataset = await data_agent.run(state)
print(dataset.title)
for p in dataset.points:
    print(f"  {p.label:20s} {p.value}")

Approximate Share of Developers by Code Editor
  VS Code              74.0
  Visual Studio        17.0
  IntelliJ IDEA        15.0
  Vim                  11.0
  Sublime Text         6.0
  Notepad++            5.0


## 4. Wrap the data as a typed component and emit it

`ChartComponent` is a `UIComponentSpec` — a typed envelope with a `component_type` discriminator and a typed `data` payload. `UIEmitter` publishes it on the bus as a `ui.chart.init` event. Anything subscribed to the bus (a frontend, the SSE sidecar, a logger) sees it.

In [4]:
from orqest.ui import UIEmitter, ChartComponent, ChartComponentData, ChartSeries
from orqest.observability import AgentEvent

# Collect everything that crosses the bus so we can inspect it.
seen: list[AgentEvent] = []
wb.event_bus.subscribe_all(seen.append)

chart = ChartComponent(
    component_id="editor-share",
    data=ChartComponentData(
        chart_kind="bar",
        title=dataset.title,
        series=[ChartSeries(
            name=dataset.series_name,
            points=[{"x": p.label, "y": p.value} for p in dataset.points],
        )],
    ),
)

emitter = UIEmitter(wb.event_bus, agent_name="data_agent")
await emitter.init(chart)

evt = seen[-1]
print(f"event_type: {evt.event_type}")
print(f"payload keys: {sorted(evt.data.keys())}")
print(f"chart_kind:  {evt.data['data']['chart_kind']}")
print(f"points:      {len(evt.data['data']['series'][0]['points'])}")

event_type: ui.chart.init
payload keys: ['component_id', 'component_type', 'created_at', 'data', 'metadata']
chart_kind:  bar
points:      6


## 5. Streaming updates — `ExecutionPlan` as a live component

Some components change over time. `ExecutionPlan` models a task list; call `enable_ui_events()` and every `set_task_status` emits a typed `ui.plan.delta` *alongside* the legacy `plan.task.updated` event — so a generative-UI frontend and a plain SSE consumer both stay in sync. `emit_init` sends the initial `ui.plan.init`.

In [5]:
from orqest.plan import ExecutionPlan, PlanTask

plan = ExecutionPlan(tasks=[
    PlanTask(id="t1", title="Generate the dataset"),
    PlanTask(id="t2", title="Render the chart"),
]).enable_ui_events(component_id="render-plan")

await plan.emit_init(wb.event_bus, agent_name="ui_agent")
await plan.set_task_status("t1", "completed", bus=wb.event_bus, agent_name="ui_agent")
await plan.set_task_status("t2", "in-progress", bus=wb.event_bus, agent_name="ui_agent")

# Both the legacy plan.* channel and the typed ui.plan.* channel fire.
for e in seen:
    if e.event_type.startswith(("plan.", "ui.plan.")):
        print(f"{e.event_type:22s} {e.data}")

plan.init              {'tasks': [{'id': 't1', 'title': 'Generate the dataset', 'description': '', 'status': 'pending', 'priority': 'required', 'level': 0, 'dependencies': [], 'subtasks': []}, {'id': 't2', 'title': 'Render the chart', 'description': '', 'status': 'pending', 'priority': 'required', 'level': 0, 'dependencies': [], 'subtasks': []}]}
ui.plan.init           {'component_type': 'plan', 'component_id': 'render-plan', 'data': {'tasks': [{'id': 't1', 'title': 'Generate the dataset', 'description': '', 'status': 'pending', 'priority': 'required', 'level': 0, 'dependencies': [], 'subtasks': []}, {'id': 't2', 'title': 'Render the chart', 'description': '', 'status': 'pending', 'priority': 'required', 'level': 0, 'dependencies': [], 'subtasks': []}]}, 'metadata': {}, 'created_at': '2026-05-16T09:01:29.381503Z'}
plan.task.updated      {'task_id': 't1', 'status': 'completed'}
ui.plan.delta          {'component_id': 'render-plan', 'component_type': 'plan', 'op': 'replace', 'path': 'tas

## 6. The SSE sidecar — the bus as a stream

`sse_sidecar` subscribes to the bus and yields SSE-formatted strings — drop it straight into a FastAPI `StreamingResponse`. It subscribes *eagerly*, so events emitted after you create it are captured. Here we create the sidecar, emit two fresh component events, and pull the raw wire format back out.

In [6]:
from orqest.observability import sse_sidecar
from orqest.ui import TableComponent, TableComponentData, TableColumn

sidecar = sse_sidecar(wb.event_bus, heartbeat_s=30.0)

# Emit two components *after* the sidecar is subscribed.
await emitter.init(chart)
table = TableComponent(
    component_id="editor-table",
    data=TableComponentData(
        columns=[TableColumn(key="label", label="Editor"),
                 TableColumn(key="value", label="Share")],
        rows=[{"label": p.label, "value": p.value} for p in dataset.points],
    ),
)
await emitter.init(table)

chunks = []
async for chunk in sidecar:
    chunks.append(chunk)
    if len(chunks) >= 2:
        break

for chunk in chunks:
    print(chunk.split(chr(10))[0])  # the "event:" line
    print(f"  ({len(chunk)} bytes of SSE)")

event: ui.chart.init
  (668 bytes of SSE)
event: ui.table.init
  (704 bytes of SSE)


## What's happening under the hood

- A `UIComponentSpec` subclass (`ChartComponent`, `TableComponent`, …) is just a Pydantic model with a `component_type` `Literal` discriminator and a typed `data` payload. Third parties register their own — the registry is per-`Workbench`, not a global singleton.
- `UIEmitter` is a thin facade: `init` / `delta` / `remove` map to the `ui.<type>.{init,delta,remove}` event-type convention. Bus failures are logged, never raised — UI emission can't break agent execution.
- `ExecutionPlan.enable_ui_events()` is opt-in *dual emission*: the legacy `plan.*` events keep firing byte-for-byte, and the typed `ui.plan.*` events fire alongside — so migrating a frontend is incremental.
- `sse_sidecar` ring-buffers against slow consumers and supports replay, so a reconnecting client can catch up. It is transport-agnostic: the same iterator feeds FastAPI, Starlette, or any ASGI `StreamingResponse`.
- The agent never imports a UI library. It produces typed data; the component layer is the consumer's choice of surface.